In [0]:
# Install required libraries
# azure-storage-blob: connects to Azure Blob Storage
# openpyxl: required by pandas to read .xlsx Excel files
%pip install azure-storage-blob openpyxl -q
print("Libraries installed successfully!")

In [0]:
# Restart kernel after install to load new libraries
dbutils.library.restartPython()

##  Connect & load synergysuite raw file

In [0]:
# ============================================
# Connect to Azure Blob Storage
# Load SynergySuite raw Excel from bronze
# ============================================
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io

# Connect securely via Key Vault — never hardcode credentials
storage_account_key = dbutils.secrets.get(scope="churchs-payroll-kv", key="storage-account-key")
blob_service_client = BlobServiceClient(
    account_url="https://churchspayrollstorage.blob.core.windows.net",
    credential=storage_account_key
)
print("Connected to Azure Blob Storage successfully!")

# Point to SynergySuite file in bronze container
# Note: folder name has trailing space — 'synergysuite '
blob_client = blob_service_client.get_blob_client(
    container="bronze",
    blob="synergysuite /SynergySuite Input.xlsx"
)

# Download and read Excel without headers
# header=None because file has complex multi-row header structure
blob_data = blob_client.download_blob().readall()
df_raw = pd.read_excel(io.BytesIO(blob_data), header=None)

# Validation — check file loaded correctly
print("Raw file loaded! Shape:", df_raw.shape)
assert df_raw.shape[0] > 0, "ERROR: Raw file is empty!"
assert df_raw.shape[1] > 0, "ERROR: No columns found!"
print("Validation passed — file has", df_raw.shape[0], "rows and", df_raw.shape[1], "columns")

## Column Index Mapping

In [0]:
# ============================================
# AUTO DETECT: Column Position Discovery
# Purpose: Auto detect column positions
# from raw Excel without manual inspection
# ============================================

import logging
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(message)s")
logger = logging.getLogger(__name__)

def detect_column_positions(df, sample_data_row=8):
    """
    Auto detects column positions from raw Excel.
    Returns dictionary of column_name: column_index
    """
    positions = {}
    
    # Combine header rows 0 and 1
    combined_headers = {}
    for h_row in [0, 1]:
        for col_idx in range(df.shape[1]):
            val = str(df.iloc[h_row, col_idx]).strip() if pd.notna(df.iloc[h_row, col_idx]) else ""
            val = val.replace("\n", " ").strip()
            if val and val != "nan":
                if col_idx in combined_headers:
                    combined_headers[col_idx] += f" {val}"
                else:
                    combined_headers[col_idx] = val

    # Map known column names to positions
    for col_idx, header in combined_headers.items():
        header_lower = header.lower()
        if "shift date" in header_lower:
            positions["Shift_Date"] = col_idx
        elif "clock in" in header_lower:
            positions["Clock_In"] = col_idx
        elif "clock out" in header_lower:
            positions["Clock_Out"] = col_idx
        elif "regular" in header_lower and "hours" not in combined_headers.get(col_idx-1, "").lower():
            positions["Regular_Hours"] = col_idx
        elif "overtime" in header_lower or "over time" in header_lower:
            positions["OT_Hours"] = col_idx
        elif "doubletime" in header_lower or "double time" in header_lower:
            positions["Double_Hours"] = col_idx
        elif "non-cash tips" in header_lower or "non cash tips" in header_lower:
            positions["Non_Cash_Tips"] = col_idx
        elif "declared tips" in header_lower:
            positions["Declared_Tips"] = col_idx

    # Detect labor columns — they come right after hours columns
    if "Regular_Hours" in positions:
        positions["Regular_Labor"] = positions["Regular_Hours"] + 2
    if "OT_Hours" in positions:
        positions["OT_Labor"] = positions["OT_Hours"] + 1
    if "Double_Hours" in positions:
        positions["Double_Labor"] = positions["Double_Hours"] + 1

    # Total Hours and Total Labor
    if "Double_Labor" in positions:
        base = positions["Double_Labor"]
        positions["Total_Hours"]   = base + 1
        positions["Total_Labor"]   = base + 2
        positions["Paid_Breaks"]   = base + 3
        positions["Unpaid_Breaks"] = base + 4

    # Rate — first numeric value in data row not matching other columns
    sample_row = df.iloc[sample_data_row]
    for col_idx in range(df.shape[1]):
        if col_idx not in positions.values():
            val = sample_row[col_idx]
            if pd.notna(val) and str(val).replace('.','').isdigit():
                if col_idx not in [positions.get("Regular_Hours"),
                                   positions.get("Regular_Labor"),
                                   positions.get("OT_Hours")]:
                    positions["Rate"] = col_idx
                    break

    return positions

# Run auto detection
logger.info("Starting auto column detection...")
col_pos = detect_column_positions(df_raw, sample_data_row=8)

print("\n=== Auto Detected Column Positions ===")
for name, idx in col_pos.items():
    sample = str(df_raw.iloc[8, idx]).strip() if pd.notna(df_raw.iloc[8, idx]) else "NaN"
    print(f"  {name:<20} → row[{idx}] = {sample}")
logger.info("Auto detection complete!")

## Parse hierarchy 

In [0]:
# ============================================
# Parse SynergySuite hierarchy
# Structure: Store → Employee → Position → Shift rows
# Uses auto detected column positions from Cell 4
# ============================================

logger.info("Starting SynergySuite hierarchy parsing using auto detected columns...")
logger.info(f"Total raw rows to process: {len(df_raw)}")

records = []
store = employee = payroll = position = None

store_count = 0
employee_count = 0
skipped_rows = 0

for idx, row in df_raw.iterrows():
    b = str(row[1]).strip() if pd.notna(row[1]) else ""

    # Store row — identified by FC and dash in name
    # Reset all context variables when new store is found
    if "FC" in b and "-" in b:
        store = b
        employee = None
        payroll = None
        position = None
        store_count += 1
        logger.info(f"Store found: {store}")

    # Employee row — identified by Employee / Payroll label
    elif "Employee / Payroll" in b:
        info = str(row[5]).strip() if pd.notna(row[5]) else ""
        if "/" in info:
            employee, payroll = [x.strip() for x in info.split("/")]
            employee_count += 1
            logger.info(f"Employee found: {employee} | Payroll ID: {payroll} | Store: {store}")

    # Position row
    elif b == "Position":
        position = str(row[2]).strip() if pd.notna(row[2]) else ""
        logger.info(f"Position found: {position} | Employee: {employee}")

    # Shift data row — identified by numeric rate in auto detected Rate column
    elif pd.notna(row[col_pos.get("Rate", 3)]) and str(row[col_pos.get("Rate", 3)]).replace('.','').isdigit():
        try:
            records.append({
                "StoreID":        store,
                "Employee_name":  employee,
                "Payroll_ID":     payroll,
                "Position":       position,
                "Shift_Date":     pd.to_datetime(b, errors="coerce"),
                "Rate":           row[col_pos.get("Rate",          3)],
                "Clock_In":       row[col_pos.get("Clock_In",      5)],
                "Clock_Out":      row[col_pos.get("Clock_Out",     6)],
                "Regular_Hours":  pd.to_numeric(row[col_pos.get("Regular_Hours",  7)], errors="coerce"),
                "Regular_Labor":  row[col_pos.get("Regular_Labor", 9)],
                "OT_Hours":       row[col_pos.get("OT_Hours",      10)],
                "OT_Labor":       row[col_pos.get("OT_Labor",      11)],
                "Double_Hours":   row[col_pos.get("Double_Hours",  12)],
                "Double_Labor":   row[col_pos.get("Double_Labor",  13)],
                "Total_Hours":    row[col_pos.get("Total_Hours",   14)],
                "Total_Labor":    row[col_pos.get("Total_Labor",   15)],
                "Paid_Breaks":    row[col_pos.get("Paid_Breaks",   16)],
                "Unpaid_Breaks":  row[col_pos.get("Unpaid_Breaks", 17)],
                "Non_Cash_Tips":  row[col_pos.get("Non_Cash_Tips", 18)],
                "Declared_Tips":  row[col_pos.get("Declared_Tips", 19)]
            })
        except Exception as e:
            skipped_rows += 1
            logger.error(f"ERROR: Failed to parse row {idx} — {e}")
            logger.error(f"Row data: {row.to_dict()}")

logger.info("Parsing loop complete!")

# Build silver DataFrame
df_silver = pd.DataFrame(records)
logger.info(f"DataFrame created with {len(df_silver)} records and {len(df_silver.columns)} columns")

# Add week ending date — Saturday of each shift week
df_silver["week_end_date"] = df_silver["Shift_Date"].apply(
    lambda d: d + pd.Timedelta(days=(5 - d.weekday()) % 7) if pd.notna(d) else None
)
logger.info("week_end_date calculated successfully")

# Logs
print("\n=== Parse Summary ===")
print("Stores found:", store_count)
print("Employees found:", employee_count)
print("Shift records parsed:", len(records))
print("Rows skipped:", skipped_rows)

# Validation checks
print("\n=== Validation Checks ===")
try:
    assert len(df_silver) > 0, "ERROR: No records parsed!"
    assert df_silver["StoreID"].notna().all(), "ERROR: Missing StoreID values!"
    assert df_silver["Employee_name"].notna().all(), "ERROR: Missing Employee names!"
    assert df_silver["Payroll_ID"].notna().all(), "ERROR: Missing Payroll IDs!"
    assert df_silver["Shift_Date"].notna().sum() > 0, "ERROR: No valid shift dates!"
    assert df_silver["Regular_Hours"].notna().sum() > 0, "ERROR: No valid Regular Hours!"
    logger.info("All validation checks passed!")
    print("All validation checks passed!")
except AssertionError as e:
    logger.error(f"VALIDATION FAILED: {e}")
    raise

print("\nUnique stores:", df_silver["StoreID"].nunique())
print("Unique employees:", df_silver["Payroll_ID"].nunique())
print("Date range:", df_silver["Shift_Date"].min(), "to", df_silver["Shift_Date"].max())
print("Regular Hours range:", df_silver["Regular_Hours"].min(), "to", df_silver["Regular_Hours"].max())
print("\nSample:")
print(df_silver.head(5).to_string())

## SynergySuite Bronze → Silver

In [0]:
# ============================================
# Upload cleaned SynergySuite data
# to silver container as CSV
# ============================================

logger.info("Starting upload of SynergySuite silver CSV to blob storage...")

# Convert DataFrame to CSV in memory
csv_buffer = io.BytesIO()
df_silver.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)
logger.info(f"CSV buffer created — {len(df_silver)} records ready for upload")

# Upload to silver container
silver_client = blob_service_client.get_blob_client(
    container="silver",
    blob="synergysuite/SynergySuite_Silver.csv"
)

try:
    silver_client.upload_blob(csv_buffer, overwrite=True)
    logger.info("SynergySuite Silver CSV uploaded successfully!")
    logger.info("Location: silver/synergysuite/SynergySuite_Silver.csv")
except Exception as e:
    logger.error(f"ERROR: Failed to upload SynergySuite Silver CSV — {e}")
    raise

# Validation — verify upload
try:
    assert len(df_silver) > 0, "ERROR: Nothing was written to silver!"
    logger.info("Upload validation passed!")
    print("\nSynergySuite Silver CSV uploaded successfully!")
    print("Location: silver/synergysuite/SynergySuite_Silver.csv")
    print("Total records written:", len(df_silver))
    print("Columns written:", df_silver.columns.tolist())
    print("Upload validation passed!")
except AssertionError as e:
    logger.error(f"UPLOAD VALIDATION FAILED: {e}")
    raise

In [0]:
df_silver.head